# Building Agentic AutoML

## Goal

This notebook is a hands-on journey to build an Agentic AutoML system from scratch.

Each version introduces one new concept, allowing the agent to evolve step by step while practicing AutoML and agent development.

---

## Version 8

In this version, we extend the system with a **Senior Agent** responsible for autonomous experiment planning and decision making.

Instead of evaluating every possible preprocessing, feature engineering, model, and hyperparameter combination exhaustively, the agent now uses dataset information and previous experiment results to decide which experiments are worth running next.

The Senior Agent can prioritize promising candidates, avoid unnecessary experiments, and allocate a limited experiment budget while keeping a transparent history of its decisions.

The preprocessing, feature engineering, model selection, hyperparameter configurations, metrics, and validation logic introduced in previous versions remain available as tools that the agent can choose from.

Version 8 focuses on one new concept only: **Autonomous Experiment Planning**.

## 1. Imports

In [1]:
from time import perf_counter

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor

from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

## 2. Data

We continue using the Adult Income dataset as the primary development benchmark so that the main new concept introduced in Version 8 remains autonomous experiment planning.

Keeping the same dataset also allows us to compare the Senior Agent directly with the exhaustive search performed in Version 7.

The dataset provides numerical and categorical features, missing values, class imbalance, and enough complexity for the agent to make meaningful experiment-planning decisions.

Additional datasets will be used later to evaluate whether the final agent can generalize to multiclass classification and regression problems.

The target column is provided by the user.

In [2]:
DATA_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/adult_income.csv"
TARGET = "income"

df = pd.read_csv(DATA_PATH)

df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Task Detection

The agent first inspects the target to identify the machine learning task.

Version 8 extends the previous task-detection logic so that the agent can distinguish between:

- binary classification
- multiclass classification
- regression

Non-numerical targets are treated as classification targets.

For numerical targets, the agent considers both the number of unique values and their proportion relative to the dataset size. A small set of repeated discrete values is treated as classification, while targets with many distinct numerical values are treated as regression.

The detected task can later be used by the Senior Agent to select suitable metrics, validation strategies, models, and experiments.

In [3]:
def detect_task(df, target):
    y = df[target]
    n_unique = y.nunique(dropna=True)
    unique_ratio = n_unique / len(y)

    if not pd.api.types.is_numeric_dtype(y):
        task = "classification"
    elif n_unique <= 20 and unique_ratio <= 0.05:
        task = "classification"
    else:
        task = "regression"

    if task == "classification":
        classification_type = "binary" if n_unique == 2 else "multiclass"
    else:
        classification_type = None

    return task, classification_type

In [4]:
task, classification_type = detect_task(df, TARGET)

task, classification_type

('classification', 'binary')

## 4. Feature Detection

The agent identifies numerical and categorical features.

For now, feature types are detected directly from the dataframe dtypes.

In [5]:
def detect_features(df, target):
    X = df.drop(columns=target)

    numerical = X.select_dtypes(include="number").columns.tolist()
    categorical = X.select_dtypes(exclude="number").columns.tolist()

    return numerical, categorical

In [6]:
numerical_features, categorical_features = detect_features(df, TARGET)

numerical_features, categorical_features

(['age',
  'fnlwgt',
  'education_num',
  'capital_gain',
  'capital_loss',
  'hours_per_week'],
 ['workclass',
  'education',
  'marital_status',
  'occupation',
  'relationship',
  'race',
  'sex',
  'native_country'])

## 5. Data Inspection

The agent inspects the dataset before planning any experiment.

Version 8 extends the dataset inspection with additional signals that can support autonomous decisions.

The inspection now includes:

- dataset dimensions
- target type and distribution
- feature data types
- missing-value counts and percentages
- feature cardinality
- duplicate rows
- target imbalance for classification problems

These signals are stored as part of the dataset inspection and are available to the Senior Agent. In the current planning policy, missing-value information is used directly to guide preprocessing decisions, while the remaining signals provide context for future extensions.

In [7]:
def inspect_dataset(df, target, task):
    X = df.drop(columns=target)
    y = df[target]

    target_info = {
        "name": target,
        "dtype": str(y.dtype),
        "unique_values": int(y.nunique(dropna=True)),
        "missing_values": int(y.isna().sum())
    }

    if task == "classification":
        counts = y.value_counts(dropna=False)
        target_info["distribution"] = counts.to_dict()
        target_info["distribution_ratio"] = (counts / len(y)).round(4).to_dict()
    else:
        target_info["summary"] = y.describe().to_dict()

    missing = df.isna().sum()

    return {
        "shape": {
            "rows": len(df),
            "columns": len(df.columns)
        },
        "duplicate_rows": int(df.duplicated().sum()),
        "target": target_info,
        "dtypes": {column: str(dtype) for column, dtype in X.dtypes.items()},
        "missing_values": missing.to_dict(),
        "missing_percentage": (missing / len(df) * 100).round(2).to_dict(),
        "cardinality": X.nunique(dropna=True).to_dict()
    }

In [8]:
inspection = inspect_dataset(df, TARGET, task)

inspection

{'shape': {'rows': 48842, 'columns': 15},
 'duplicate_rows': 52,
 'target': {'name': 'income',
  'dtype': 'object',
  'unique_values': 2,
  'missing_values': 0,
  'distribution': {'<=50K': 37155, '>50K': 11687},
  'distribution_ratio': {'<=50K': 0.7607, '>50K': 0.2393}},
 'dtypes': {'age': 'int64',
  'workclass': 'object',
  'fnlwgt': 'int64',
  'education': 'object',
  'education_num': 'int64',
  'marital_status': 'object',
  'occupation': 'object',
  'relationship': 'object',
  'race': 'object',
  'sex': 'object',
  'capital_gain': 'int64',
  'capital_loss': 'int64',
  'hours_per_week': 'int64',
  'native_country': 'object'},
 'missing_values': {'age': 0,
  'workclass': 2799,
  'fnlwgt': 0,
  'education': 0,
  'education_num': 0,
  'marital_status': 0,
  'occupation': 2809,
  'relationship': 0,
  'race': 0,
  'sex': 0,
  'capital_gain': 0,
  'capital_loss': 0,
  'hours_per_week': 0,
  'native_country': 857,
  'income': 0},
 'missing_percentage': {'age': 0.0,
  'workclass': 5.73,
  'f

## 6. Preprocessing

The preprocessing strategies introduced in Version 3 remain available to the Senior Agent:

- `native`: preserve missing values whenever the selected model supports them directly
- `impute`: fill missing numerical values with the training median and categorical values with the most frequent training category

Preprocessing parameters are always learned exclusively from the training portion of each cross-validation fold.

Version 8 also makes the preprocessing step more robust to edge cases such as completely missing numerical or categorical columns.

The Senior Agent will later decide which preprocessing strategies are worth evaluating based on the dataset inspection and previous experiment results.

In [9]:
def prepare_data(df, target):
    X = df.drop(columns=target).copy()
    y = df[target].copy()

    return X, y

In [10]:
X, y = prepare_data(df, TARGET)

X.shape, y.shape

((48842, 14), (48842,))

In [11]:
def preprocess_data(X_train, X_valid, numerical_features, categorical_features, strategy):
    X_train = X_train.copy()
    X_valid = X_valid.copy()

    if strategy == "impute":
        for column in numerical_features:
            value = X_train[column].median()

            if pd.isna(value):
                value = 0

            X_train[column] = X_train[column].fillna(value)
            X_valid[column] = X_valid[column].fillna(value)

        for column in categorical_features:
            mode = X_train[column].mode(dropna=True)
            value = mode.iloc[0] if not mode.empty else "__MISSING__"

            X_train[column] = X_train[column].fillna(value)
            X_valid[column] = X_valid[column].fillna(value)

    for column in categorical_features:
        categories = X_train[column].dropna().unique().tolist()

        if not categories:
            categories = ["__MISSING__"]

        dtype = pd.CategoricalDtype(categories=categories)

        X_train[column] = X_train[column].astype(dtype)
        X_valid[column] = X_valid[column].astype(dtype)

    return X_train, X_valid

In [12]:
PREPROCESSING_STRATEGIES = ["native", "impute"]

PREPROCESSING_STRATEGIES

['native', 'impute']

## 7. Feature Engineering

The feature engineering strategies introduced in Version 6 remain available to the Senior Agent:

- `none`: keep the original feature space unchanged
- `interactions`: add pairwise multiplication features between numerical variables

Feature engineering is applied independently inside each cross-validation fold after preprocessing, and the original features are always preserved.

Unlike Version 7, Version 8 does not necessarily evaluate every feature engineering strategy for every candidate.

The Senior Agent can use dataset characteristics and previous experiment results to decide whether additional feature engineering experiments are worth the computational cost.

In [13]:
FEATURE_ENGINEERING_STRATEGIES = ["none", "interactions"]


def apply_feature_engineering(X_train, X_valid, numerical_features, strategy):
    X_train = X_train.copy()
    X_valid = X_valid.copy()

    if strategy == "interactions":
        for i in range(len(numerical_features)):
            for j in range(i + 1, len(numerical_features)):
                feature_a = numerical_features[i]
                feature_b = numerical_features[j]

                new_feature = f"{feature_a}_x_{feature_b}"

                X_train[new_feature] = X_train[feature_a] * X_train[feature_b]
                X_valid[new_feature] = X_valid[feature_a] * X_valid[feature_b]

    return X_train, X_valid

## 8. Model Selection

The same three boosting model families introduced in previous versions remain available to the Senior Agent:

- LightGBM
- XGBoost
- CatBoost

Version 8 extends model initialization so that each classifier is configured appropriately for binary or multiclass classification.

Regression models remain unchanged.

Unlike Version 7, the Senior Agent will not necessarily evaluate every model under every possible pipeline configuration. Each model family is first evaluated as a baseline candidate, after which previous experiment results and the remaining budget influence which model configurations are explored further.

In [14]:
def select_models(task, classification_type, categorical_features, model_params=None):
    model_params = model_params or {}

    lightgbm_params = model_params.get("LightGBM", {})
    xgboost_params = model_params.get("XGBoost", {})
    catboost_params = model_params.get("CatBoost", {})

    if task == "classification":
        lightgbm_base = {
            "random_state": 42,
            "verbosity": -1,
            "objective": "binary" if classification_type == "binary" else "multiclass"
        }

        xgboost_base = {
            "random_state": 42,
            "tree_method": "hist",
            "enable_categorical": True,
            "verbosity": 0,
            "objective": "binary:logistic" if classification_type == "binary" else "multi:softprob"
        }

        catboost_base = {
            "random_seed": 42,
            "cat_features": categorical_features,
            "verbose": False,
            "allow_writing_files": False,
            "loss_function": "Logloss" if classification_type == "binary" else "MultiClass"
        }

        lightgbm_base.update(lightgbm_params)
        xgboost_base.update(xgboost_params)
        catboost_base.update(catboost_params)

        return {
            "LightGBM": LGBMClassifier(**lightgbm_base),
            "XGBoost": XGBClassifier(**xgboost_base),
            "CatBoost": CatBoostClassifier(**catboost_base)
        }

    lightgbm_base = {"random_state": 42, "verbosity": -1}
    xgboost_base = {
        "random_state": 42,
        "tree_method": "hist",
        "enable_categorical": True,
        "verbosity": 0
    }
    catboost_base = {
        "random_seed": 42,
        "cat_features": categorical_features,
        "verbose": False,
        "allow_writing_files": False
    }

    lightgbm_base.update(lightgbm_params)
    xgboost_base.update(xgboost_params)
    catboost_base.update(catboost_params)

    return {
        "LightGBM": LGBMRegressor(**lightgbm_base),
        "XGBoost": XGBRegressor(**xgboost_base),
        "CatBoost": CatBoostRegressor(**catboost_base)
    }

In [15]:
models = select_models(task, classification_type, categorical_features)

models

{'LightGBM': LGBMClassifier(objective='binary', random_state=42, verbosity=-1),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=None, num_parallel_tree=None, ...),
 'CatBoost': CatBoostClassifier(allow_writing_files=False, cat_features=['workclass', 'education', 'marital_status', 'occupation', 'relat

## 9. Experiment Space

Version 8 keeps the compact hyperparameter configurations introduced in Version 7, but changes how they are used.

Instead of evaluating the complete Cartesian product exhaustively, preprocessing strategies, feature engineering strategies, model families, and hyperparameter configurations now define a pool of candidate experiments.

Each candidate experiment specifies:

- preprocessing strategy
- feature engineering strategy
- model family
- hyperparameter configuration

The Senior Agent will later decide which candidates should actually be executed and in which order.

This separates the available experiment space from the experiment-planning policy.

In [16]:
HYPERPARAMETER_SEARCH_SPACES = {
    "LightGBM": [
        {},
        {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 31}
    ],
    "XGBoost": [
        {},
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 4}
    ],
    "CatBoost": [
        {},
        {"iterations": 500, "learning_rate": 0.05, "depth": 6}
    ]
}

HYPERPARAMETER_SEARCH_SPACES

{'LightGBM': [{},
  {'n_estimators': 200, 'learning_rate': 0.05, 'num_leaves': 31}],
 'XGBoost': [{}, {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4}],
 'CatBoost': [{}, {'iterations': 500, 'learning_rate': 0.05, 'depth': 6}]}

In [17]:
def build_experiment_space(model_names):
    return [
        {
            "preprocessing": preprocessing,
            "feature_engineering": feature_engineering,
            "model": model_name,
            "hyperparameters": dict(model_params)
        }
        for preprocessing in PREPROCESSING_STRATEGIES
        for feature_engineering in FEATURE_ENGINEERING_STRATEGIES
        for model_name in model_names
        for model_params in HYPERPARAMETER_SEARCH_SPACES[model_name]
    ]


experiment_space = build_experiment_space(models.keys())

len(experiment_space), experiment_space[:3]

(24,
 [{'preprocessing': 'native',
   'feature_engineering': 'none',
   'model': 'LightGBM',
   'hyperparameters': {}},
  {'preprocessing': 'native',
   'feature_engineering': 'none',
   'model': 'LightGBM',
   'hyperparameters': {'n_estimators': 200,
    'learning_rate': 0.05,
    'num_leaves': 31}},
  {'preprocessing': 'native',
   'feature_engineering': 'none',
   'model': 'XGBoost',
   'hyperparameters': {}}])

## 10. Metric Selection

The evaluation metric is selected according to the detected task.

The agent uses:

- `ROC AUC` for binary classification
- `ROC AUC OvR Macro` for multiclass classification
- `RMSE` for regression

For multiclass problems, ROC AUC is computed using the one-vs-rest strategy and macro averaging so that each class contributes equally to the final score.

The selected metric becomes part of the information used by the Senior Agent to compare experiments consistently.

In [18]:
def select_metric(task, classification_type):
    if task == "classification":
        if classification_type == "binary":
            return "roc_auc"

        return "roc_auc_ovr_macro"

    return "rmse"

In [19]:
metric = select_metric(task, classification_type)

metric

'roc_auc'

## 11. Validation Strategy

The validation strategy is selected according to the detected task.

The agent uses:

- `StratifiedKFold` for classification
- `KFold` for regression

The default validation budget is five folds.

For classification problems, Version 8 also checks the size of the smallest target class. If a class contains fewer than five observations, the number of folds is automatically reduced so that every validation split can still contain examples from each class.

At least two observations per class are required for cross-validation.

This makes the validation layer more robust while preserving the same evaluation principles introduced in previous versions.

In [20]:
def select_validation(task, y, max_splits=5):
    if task == "classification":
        min_class_count = int(y.value_counts().min())

        if min_class_count < 2:
            raise ValueError("Each class must contain at least 2 observations for cross-validation.")

        n_splits = min(max_splits, min_class_count)

        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    n_splits = min(max_splits, len(y))

    if n_splits < 2:
        raise ValueError("At least 2 observations are required for cross-validation.")

    return KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [21]:
validation = select_validation(task, y)

validation

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

## 12. Training and Evaluation

The training and evaluation layer executes a single candidate experiment across the selected cross-validation folds.

Inside each fold:

- preprocessing is learned from the training data
- feature engineering is applied after preprocessing
- a fresh model instance is created
- classification targets are encoded consistently within the fold
- the model is trained and evaluated using the selected metric

Version 8 extends the evaluation layer with:

- binary and multiclass classification support
- experiment execution time tracking
- explicit experiment status
- failure handling without stopping the entire AutoML process

These additions allow the Senior Agent to record model quality, execution cost, and experiment failures while maintaining a transparent experiment history.

In [22]:
def train_and_evaluate(
    X,
    y,
    model_name,
    task,
    classification_type,
    numerical_features,
    categorical_features,
    preprocessing,
    feature_engineering,
    model_params,
    validation
):
    start_time = perf_counter()
    fold_scores = []

    try:
        for train_idx, valid_idx in validation.split(X, y):
            X_train = X.iloc[train_idx].copy()
            X_valid = X.iloc[valid_idx].copy()

            y_train = y.iloc[train_idx].copy()
            y_valid = y.iloc[valid_idx].copy()

            X_train, X_valid = preprocess_data(
                X_train,
                X_valid,
                numerical_features,
                categorical_features,
                preprocessing
            )

            X_train, X_valid = apply_feature_engineering(
                X_train,
                X_valid,
                numerical_features,
                feature_engineering
            )

            fold_model = select_models(
                task,
                classification_type,
                categorical_features,
                {model_name: model_params}
            )[model_name]

            y_fit = y_train
            y_eval = y_valid

            if task == "classification":
                classes = sorted(y_train.dropna().unique())
                mapping = {label: index for index, label in enumerate(classes)}

                y_fit = y_train.map(mapping)
                y_eval = y_valid.map(mapping)

                if y_eval.isna().any():
                    raise ValueError("Validation fold contains a class not present in the training fold.")

            if model_name == "CatBoost":
                for column in categorical_features:
                    X_train[column] = (
                        X_train[column]
                        .astype("object")
                        .fillna("__MISSING__")
                        .astype(str)
                    )

                    X_valid[column] = (
                        X_valid[column]
                        .astype("object")
                        .fillna("__MISSING__")
                        .astype(str)
                    )

            fold_model.fit(X_train, y_fit)

            if task == "classification":
                predictions = fold_model.predict_proba(X_valid)

                if classification_type == "binary":
                    score = roc_auc_score(y_eval, predictions[:, 1])
                else:
                    score = roc_auc_score(
                        y_eval,
                        predictions,
                        multi_class="ovr",
                        average="macro"
                    )
            else:
                predictions = fold_model.predict(X_valid)
                score = mean_squared_error(y_valid, predictions) ** 0.5

            fold_scores.append(float(score))

        return {
            "status": "completed",
            "fold_scores": fold_scores,
            "mean_score": float(np.mean(fold_scores)),
            "std_score": float(np.std(fold_scores)),
            "duration_seconds": float(perf_counter() - start_time),
            "error": None
        }

    except Exception as error:
        return {
            "status": "failed",
            "fold_scores": fold_scores,
            "mean_score": None,
            "std_score": None,
            "duration_seconds": float(perf_counter() - start_time),
            "error": f"{type(error).__name__}: {error}"
        }

## 13. Senior Agent Planning

Version 8 introduces a Senior Agent that plans experiments sequentially instead of evaluating the entire experiment space exhaustively.

The agent receives:

- dataset inspection
- available candidate experiments
- previous experiment results
- the evaluation metric
- a fixed experiment budget

The planning policy follows a simple adaptive strategy:

1. establish baseline performance across the available model families
2. identify the most promising model from completed experiments
3. prioritize hyperparameter alternatives for promising models
4. explore alternative preprocessing when missing values are present
5. evaluate feature interactions only after stronger baseline configurations have been explored

Every decision is recorded together with its reason.

The objective is not to guarantee the globally best configuration, but to use the available experiment budget more intelligently than exhaustive search.

In [23]:
MAX_EXPERIMENTS = 10


def best_completed_experiment(experiments, metric):
    completed = [
        experiment for experiment in experiments
        if experiment["status"] == "completed"
    ]

    if not completed:
        return None

    key = lambda experiment: experiment["mean_score"]

    return (
        min(completed, key=key)
        if metric == "rmse"
        else max(completed, key=key)
    )


def choose_next_experiment(remaining, experiments, inspection, metric, model_names):
    completed = [
        experiment for experiment in experiments
        if experiment["status"] == "completed"
    ]

    tested_models = {
        experiment["model"]
        for experiment in completed
        if (
            experiment["preprocessing"] == "native"
            and experiment["feature_engineering"] == "none"
            and experiment["hyperparameters"] == {}
        )
    }

    has_missing = any(
        count > 0
        for column, count in inspection["missing_values"].items()
        if column != inspection["target"]["name"]
    )

    # Establish one baseline for every model family.
    for model_name in model_names:
        if model_name not in tested_models:
            for candidate in remaining:
                if (
                    candidate["model"] == model_name
                    and candidate["preprocessing"] == "native"
                    and candidate["feature_engineering"] == "none"
                    and candidate["hyperparameters"] == {}
                ):
                    return candidate, (
                        f"Establish baseline performance for {model_name}."
                    )

    best = best_completed_experiment(experiments, metric)

    if best is None:
        return remaining[0], (
            "No successful experiment is available yet."
        )

    # Try the alternative hyperparameter configuration
    # of the currently best model.
    for candidate in remaining:
        if (
            candidate["model"] == best["model"]
            and candidate["preprocessing"] == best["preprocessing"]
            and candidate["feature_engineering"] == best["feature_engineering"]
            and candidate["hyperparameters"] != {}
        ):
            return candidate, (
                f"{best['model']} is currently the strongest model; "
                "evaluate its alternative hyperparameter configuration."
            )

    # Test alternative preprocessing when missing values exist.
    if has_missing:
        alternative = (
            "impute"
            if best["preprocessing"] == "native"
            else "native"
        )

        for candidate in remaining:
            if (
                candidate["model"] == best["model"]
                and candidate["preprocessing"] == alternative
                and candidate["feature_engineering"]
                == best["feature_engineering"]
                and candidate["hyperparameters"]
                == best["hyperparameters"]
            ):
                return candidate, (
                    "Missing values are present; test whether the "
                    "alternative preprocessing strategy improves the "
                    "current best configuration."
                )

    # Test feature interactions for the current best model.
    for candidate in remaining:
        if (
            candidate["model"] == best["model"]
            and candidate["preprocessing"] == best["preprocessing"]
            and candidate["feature_engineering"] == "interactions"
            and candidate["hyperparameters"]
            == best["hyperparameters"]
        ):
            return candidate, (
                f"Evaluate numerical interactions around the current "
                f"best {best['model']} configuration."
            )

    # Fallback: continue exploring the remaining space.
    return remaining[0], ("Explore an untested candidate within the remaining budget.")

## 14. Adaptive Experiment Execution

The Senior Agent now executes experiments sequentially.

At each step, the agent:

1. observes the experiments completed so far
2. selects the next candidate according to the planning policy
3. records the reason for the decision
4. executes the selected experiment
5. records its score, execution time, and status
6. updates the current best solution

The process continues until the experiment budget is exhausted or no candidate experiments remain.

This creates an explicit decision history that makes the agent's behavior inspectable and reproducible.

In [24]:
def run_senior_agent(
    X,
    y,
    task,
    classification_type,
    numerical_features,
    categorical_features,
    inspection,
    metric,
    validation,
    experiment_space,
    model_names,
    max_experiments=10
):
    remaining = [
        {
            **candidate,
            "hyperparameters": dict(candidate["hyperparameters"])
        }
        for candidate in experiment_space
    ]

    experiments = []
    decision_history = []

    while remaining and len(experiments) < max_experiments:
        candidate, reason = choose_next_experiment(
            remaining,
            experiments,
            inspection,
            metric,
            model_names
        )

        remaining.remove(candidate)

        result = train_and_evaluate(
            X,
            y,
            candidate["model"],
            task,
            classification_type,
            numerical_features,
            categorical_features,
            candidate["preprocessing"],
            candidate["feature_engineering"],
            candidate["hyperparameters"],
            validation
        )

        experiment = {**candidate, **result}

        experiments.append(experiment)

        best = best_completed_experiment(experiments, metric)

        decision_history.append({
            "step": len(experiments),
            "reason": reason,
            "preprocessing": candidate["preprocessing"],
            "feature_engineering": candidate["feature_engineering"],
            "model": candidate["model"],
            "hyperparameters": candidate["hyperparameters"],
            "status": result["status"],
            "score": result["mean_score"],
            "duration_seconds": result["duration_seconds"],
            "best_score_after_step": (
                best["mean_score"] if best is not None else None
            )
        })

    return {
        "experiments": experiments,
        "decision_history": decision_history,
        "best_experiment": best_completed_experiment(experiments, metric),
        "experiments_executed": len(experiments),
        "experiments_available": len(experiment_space)
    }

In [25]:
senior_run = run_senior_agent(
    X,
    y,
    task,
    classification_type,
    numerical_features,
    categorical_features,
    inspection,
    metric,
    validation,
    experiment_space,
    list(models.keys()),
    max_experiments=MAX_EXPERIMENTS
)

senior_run["experiments_executed"], senior_run["experiments_available"]

(10, 24)

## 15. Decision History and Results

The Senior Agent records every experiment decision together with its motivation and observed outcome.

The decision history makes it possible to inspect:

- which experiment was selected at each step
- why the experiment was selected
- whether execution succeeded or failed
- the validation score obtained
- the execution time
- how the best observed score evolved over time

The completed experiments can also be ranked independently from the order in which they were executed.

In [26]:
decision_history_df = pd.DataFrame(senior_run["decision_history"])

decision_history_df

,step,reason,preprocessing,feature_engineering,model,hyperparameters,status,score,duration_seconds,best_score_after_step
0,1,Establish baseline performance for LightGBM.,native,none,LightGBM,{},completed,0.929556,3.494629,0.929556
1,2,Establish baseline performance for XGBoost.,native,none,XGBoost,{},completed,0.926851,4.447527,0.929556
2,3,Establish baseline performance for CatBoost.,native,none,CatBoost,{},completed,0.930631,244.104396,0.930631
3,4,CatBoost is currently the strongest model; eva...,native,none,CatBoost,"{'iterations': 500, 'learning_rate': 0.05, 'de...",completed,0.929910,111.554985,0.930631
4,5,Missing values are present; test whether the a...,impute,none,CatBoost,{},completed,0.930119,225.847267,0.930631
5,6,Evaluate numerical interactions around the cur...,native,interactions,CatBoost,{},completed,0.929786,234.235833,0.930631
6,7,Explore an untested candidate within the remai...,native,none,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,0.929846,5.111864,0.930631
7,8,Explore an untested candidate within the remai...,native,none,XGBoost,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,0.926509,4.436513,0.930631
8,9,Explore an untested candidate within the remai...,native,interactions,LightGBM,{},completed,0.928578,3.805817,0.930631
9,10,Explore an untested candidate within the remai...,native,interactions,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,0.928841,6.435136,0.930631


In [27]:
experiments_df = pd.DataFrame(senior_run["experiments"])

completed_experiments_df = experiments_df[experiments_df["status"] == "completed"].copy()

completed_experiments_df = completed_experiments_df.sort_values(
    "mean_score",
    ascending=(metric == "rmse")
).reset_index(drop=True)

completed_experiments_df

,preprocessing,feature_engineering,model,hyperparameters,status,fold_scores,mean_score,std_score,duration_seconds,error
0,native,none,CatBoost,{},completed,"[0.9270576731075596, 0.9314256313487564, 0.932...",0.930631,0.002246,244.104396,None
1,impute,none,CatBoost,{},completed,"[0.9263586846722958, 0.9313662311457596, 0.932...",0.930119,0.002428,225.847267,None
2,native,none,CatBoost,"{'iterations': 500, 'learning_rate': 0.05, 'de...",completed,"[0.9262044858895164, 0.9309988938438942, 0.932...",0.929910,0.002476,111.554985,None
3,native,none,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,"[0.9258859004984437, 0.9306843950946944, 0.931...",0.929846,0.002751,5.111864,None
4,native,interactions,CatBoost,{},completed,"[0.9260596978947119, 0.9305695374347331, 0.931...",0.929786,0.002187,234.235833,None
5,native,none,LightGBM,{},completed,"[0.9256299673563653, 0.930524872165813, 0.9315...",0.929556,0.002508,3.494629,None
6,native,interactions,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,"[0.9243988233234207, 0.9295256306695681, 0.931...",0.928841,0.002925,6.435136,None
7,native,interactions,LightGBM,{},completed,"[0.9241074630253882, 0.9296220408827653, 0.930...",0.928578,0.002922,3.805817,None
8,native,none,XGBoost,{},completed,"[0.92372150560175, 0.927186891572412, 0.929022...",0.926851,0.002227,4.447527,None
9,native,none,XGBoost,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,"[0.9223604811830863, 0.9276445666830017, 0.927...",0.926509,0.002680,4.436513,None


In [28]:
best_experiment = senior_run["best_experiment"]

best_experiment

{'preprocessing': 'native',
 'feature_engineering': 'none',
 'model': 'CatBoost',
 'hyperparameters': {},
 'status': 'completed',
 'fold_scores': [0.9270576731075596,
  0.9314256313487564,
  0.9327287870545662,
  0.9328676483756103,
  0.9290733916199625],
 'mean_score': 0.930630626301291,
 'std_score': 0.002246310473215999,
 'duration_seconds': 244.10439567599997,
 'error': None}

## 16. Agent State

The final state summarizes both the machine learning configuration identified by the Senior Agent and the process used to reach it.

In addition to the information stored in previous versions, Version 8 records:

- classification type when applicable
- experiment budget
- number of candidate experiments
- number of experiments actually executed
- total experiment execution time
- complete decision history
- best observed configuration and validation performance

The state therefore represents not only the final AutoML result, but also the agent's decision-making process.

In [29]:
best_experiment = senior_run["best_experiment"]

total_duration = sum(
    experiment["duration_seconds"]
    for experiment in senior_run["experiments"]
)

state = {
    "task": task,
    "classification_type": classification_type,
    "metric": metric,
    "validation": validation.__class__.__name__,
    "n_splits": validation.n_splits,
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "experiment_budget": MAX_EXPERIMENTS,
    "experiments_available": senior_run["experiments_available"],
    "experiments_executed": senior_run["experiments_executed"],
    "total_duration_seconds": total_duration,
    "best_preprocessing": best_experiment["preprocessing"] if best_experiment else None,
    "best_feature_engineering": best_experiment["feature_engineering"] if best_experiment else None,
    "best_model": best_experiment["model"] if best_experiment else None,
    "best_params": best_experiment["hyperparameters"] if best_experiment else None,
    "best_score": best_experiment["mean_score"] if best_experiment else None,
    "best_std": best_experiment["std_score"] if best_experiment else None,
    "experiments": senior_run["experiments"],
    "decision_history": senior_run["decision_history"]
}

state

{'task': 'classification',
 'classification_type': 'binary',
 'metric': 'roc_auc',
 'validation': 'StratifiedKFold',
 'n_splits': 5,
 'numerical_features': ['age',
  'fnlwgt',
  'education_num',
  'capital_gain',
  'capital_loss',
  'hours_per_week'],
 'categorical_features': ['workclass',
  'education',
  'marital_status',
  'occupation',
  'relationship',
  'race',
  'sex',
  'native_country'],
 'experiment_budget': 10,
 'experiments_available': 24,
 'experiments_executed': 10,
 'total_duration_seconds': 843.4739662530001,
 'best_preprocessing': 'native',
 'best_feature_engineering': 'none',
 'best_model': 'CatBoost',
 'best_params': {},
 'best_score': 0.930630626301291,
 'best_std': 0.002246310473215999,
 'experiments': [{'preprocessing': 'native',
   'feature_engineering': 'none',
   'model': 'LightGBM',
   'hyperparameters': {},
   'status': 'completed',
   'fold_scores': [0.9256299673563653,
    0.930524872165813,
    0.9315811297628095,
    0.9323195737110039,
    0.927722610417

## 17. Agent

We now combine all components into a single Senior Agent.

Given a dataset path and target column, the agent:

1. loads and inspects the dataset
2. detects the machine learning task
3. identifies numerical and categorical features
4. selects the metric and validation strategy
5. builds the available experiment space
6. plans and executes experiments adaptively within a fixed budget
7. records every decision and experiment outcome
8. returns the best observed configuration together with the complete decision history

Unlike Version 7, the agent no longer evaluates the complete experiment space exhaustively.

Version 8 therefore transforms the previous AutoML search procedure into a budget-aware and adaptive experiment-planning system.

In [30]:
def agent(data_path, target, max_experiments=10):
    df = pd.read_csv(data_path)

    if target not in df.columns:
        raise ValueError(f"Target column '{target}' was not found.")

    if df[target].isna().any():
        raise ValueError("The target column contains missing values.")

    task, classification_type = detect_task(df, target)
    numerical, categorical = detect_features(df, target)
    inspection = inspect_dataset(df, target, task)

    X, y = prepare_data(df, target)

    metric = select_metric(task, classification_type)
    validation = select_validation(task, y)

    models = select_models(task, classification_type, categorical)

    experiment_space = build_experiment_space(models.keys())

    senior_run = run_senior_agent(
        X,
        y,
        task,
        classification_type,
        numerical,
        categorical,
        inspection,
        metric,
        validation,
        experiment_space,
        list(models.keys()),
        max_experiments=max_experiments
    )

    best = senior_run["best_experiment"]

    total_duration = sum(
        experiment["duration_seconds"]
        for experiment in senior_run["experiments"]
    )

    return {
        "task": task,
        "classification_type": classification_type,
        "numerical_features": numerical,
        "categorical_features": categorical,
        "inspection": inspection,
        "metric": metric,
        "validation": validation.__class__.__name__,
        "n_splits": validation.n_splits,
        "experiment_budget": max_experiments,
        "experiments_available": senior_run["experiments_available"],
        "experiments_executed": senior_run["experiments_executed"],
        "total_duration_seconds": total_duration,
        "experiments": senior_run["experiments"],
        "decision_history": senior_run["decision_history"],
        "best_preprocessing": best["preprocessing"] if best else None,
        "best_feature_engineering": best["feature_engineering"] if best else None,
        "best_model": best["model"] if best else None,
        "best_params": best["hyperparameters"] if best else None,
        "best_score": best["mean_score"] if best else None,
        "best_std": best["std_score"] if best else None
    }

## 18. Primary Benchmark Summary

The first Version 8 run uses Adult Income as the primary benchmark.

This summary compares the available experiment space with the number of experiments actually selected by the Senior Agent and reports the best observed solution.

The objective of Version 8 is not only to obtain a competitive validation score, but also to reduce unnecessary experimentation through adaptive planning.

The complete `agent()` function will be evaluated next on additional datasets to test whether the same decision logic generalizes beyond the development benchmark.

In [31]:
primary_summary = {
    "dataset": "Adult Income",
    "task": state["task"],
    "classification_type": state["classification_type"],
    "metric": state["metric"],
    "validation": state["validation"],
    "n_splits": state["n_splits"],
    "experiments_available": state["experiments_available"],
    "experiment_budget": state["experiment_budget"],
    "experiments_executed": state["experiments_executed"],
    "search_reduction_percent": round(
        (1 - state["experiments_executed"] / state["experiments_available"]) * 100,
        2
    ),
    "best_model": state["best_model"],
    "best_preprocessing": state["best_preprocessing"],
    "best_feature_engineering": state["best_feature_engineering"],
    "best_params": state["best_params"],
    "best_score": state["best_score"],
    "best_std": state["best_std"],
    "total_duration_seconds": state["total_duration_seconds"]
}

pd.Series(primary_summary)

dataset                        Adult Income
task                         classification
classification_type                  binary
metric                              roc_auc
validation                  StratifiedKFold
n_splits                                  5
experiments_available                    24
experiment_budget                        10
experiments_executed                     10
search_reduction_percent              58.33
best_model                         CatBoost
best_preprocessing                   native
best_feature_engineering               none
best_params                              {}
best_score                         0.930631
best_std                           0.002246
total_duration_seconds           843.473966
dtype: object

## 19. Generalization Tests

The Senior Agent was developed using Adult Income as the primary benchmark.

We now evaluate the complete `agent()` function on additional datasets that were not used to design the planning policy.

The objective is to verify whether the same autonomous workflow can generalize across different machine learning problems without changing the agent logic.

### Wine - Multiclass Classification

We begin with the Wine dataset, which introduces a new task not supported by Version 7: **multiclass classification**.

In [32]:
WINE_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/wine.csv"
WINE_TARGET = "class"

wine_state = agent(WINE_PATH, WINE_TARGET, max_experiments=MAX_EXPERIMENTS)

In [33]:
wine_summary = {
    "dataset": "Wine",
    "task": wine_state["task"],
    "classification_type": wine_state["classification_type"],
    "metric": wine_state["metric"],
    "validation": wine_state["validation"],
    "n_splits": wine_state["n_splits"],
    "experiments_available": wine_state["experiments_available"],
    "experiment_budget": wine_state["experiment_budget"],
    "experiments_executed": wine_state["experiments_executed"],
    "search_reduction_percent": round(
        (1 - wine_state["experiments_executed"] / wine_state["experiments_available"]) * 100,
        2
    ),
    "best_model": wine_state["best_model"],
    "best_preprocessing": wine_state["best_preprocessing"],
    "best_feature_engineering": wine_state["best_feature_engineering"],
    "best_params": wine_state["best_params"],
    "best_score": wine_state["best_score"],
    "best_std": wine_state["best_std"],
    "total_duration_seconds": wine_state["total_duration_seconds"]
}

pd.Series(wine_summary)

dataset                                  Wine
task                           classification
classification_type                multiclass
metric                      roc_auc_ovr_macro
validation                    StratifiedKFold
n_splits                                    5
experiments_available                      24
experiment_budget                          10
experiments_executed                       10
search_reduction_percent                58.33
best_model                           CatBoost
best_preprocessing                     native
best_feature_engineering                 none
best_params                                {}
best_score                           0.998847
best_std                             0.001814
total_duration_seconds             103.446037
dtype: object

In [34]:
pd.DataFrame(wine_state["decision_history"])

,step,reason,preprocessing,feature_engineering,model,hyperparameters,status,score,duration_seconds,best_score_after_step
0,1,Establish baseline performance for LightGBM.,native,none,LightGBM,{},completed,0.998595,0.307757,0.998595
1,2,Establish baseline performance for XGBoost.,native,none,XGBoost,{},completed,0.997483,0.368443,0.998595
2,3,Establish baseline performance for CatBoost.,native,none,CatBoost,{},completed,0.998847,9.833470,0.998847
3,4,CatBoost is currently the strongest model; eva...,native,none,CatBoost,"{'iterations': 500, 'learning_rate': 0.05, 'de...",completed,0.998847,4.959597,0.998847
4,5,Evaluate numerical interactions around the cur...,native,interactions,CatBoost,{},completed,0.998837,82.772726,0.998847
5,6,Explore an untested candidate within the remai...,native,none,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,0.998379,0.600826,0.998847
6,7,Explore an untested candidate within the remai...,native,none,XGBoost,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,0.997256,0.968752,0.998847
7,8,Explore an untested candidate within the remai...,native,interactions,LightGBM,{},completed,0.997724,0.935928,0.998847
8,9,Explore an untested candidate within the remai...,native,interactions,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,0.997280,1.473664,0.998847
9,10,Explore an untested candidate within the remai...,native,interactions,XGBoost,{},completed,0.997423,1.224874,0.998847


### California Housing - Mixed-Type Regression

The second generalization test uses the California Housing dataset with numerical features, the categorical `ocean_proximity` feature, and missing values in `total_bedrooms`.

This benchmark verifies whether the same Senior Agent can automatically transition from classification to regression while preserving the adaptive experiment-planning workflow.

It also tests whether the agent can use dataset inspection to reason about alternative preprocessing strategies when missing values are present.

In [35]:
CALIFORNIA_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/california_housing.csv"
CALIFORNIA_TARGET = "median_house_value"

california_state = agent(CALIFORNIA_PATH, CALIFORNIA_TARGET, max_experiments=MAX_EXPERIMENTS)

In [36]:
california_summary = {
    "dataset": "California Housing",
    "task": california_state["task"],
    "classification_type": california_state["classification_type"],
    "metric": california_state["metric"],
    "validation": california_state["validation"],
    "n_splits": california_state["n_splits"],
    "experiments_available": california_state["experiments_available"],
    "experiment_budget": california_state["experiment_budget"],
    "experiments_executed": california_state["experiments_executed"],
    "search_reduction_percent": round(
        (1 - california_state["experiments_executed"] / california_state["experiments_available"]) * 100,
        2
    ),
    "best_model": california_state["best_model"],
    "best_preprocessing": california_state["best_preprocessing"],
    "best_feature_engineering": california_state["best_feature_engineering"],
    "best_params": california_state["best_params"],
    "best_score": california_state["best_score"],
    "best_std": california_state["best_std"],
    "total_duration_seconds": california_state["total_duration_seconds"]
}

pd.Series(california_summary)

dataset                     California Housing
task                                regression
classification_type                       None
metric                                    rmse
validation                               KFold
n_splits                                     5
experiments_available                       24
experiment_budget                           10
experiments_executed                        10
search_reduction_percent                 58.33
best_model                            CatBoost
best_preprocessing                      impute
best_feature_engineering          interactions
best_params                                 {}
best_score                        45482.045323
best_std                           1289.982782
total_duration_seconds              265.454362
dtype: object

In [37]:
pd.DataFrame(california_state["decision_history"])

,step,reason,preprocessing,feature_engineering,model,hyperparameters,status,score,duration_seconds,best_score_after_step
0,1,Establish baseline performance for LightGBM.,native,none,LightGBM,{},completed,47518.598892,1.445870,47518.598892
1,2,Establish baseline performance for XGBoost.,native,none,XGBoost,{},completed,47337.678787,1.992868,47337.678787
2,3,Establish baseline performance for CatBoost.,native,none,CatBoost,{},completed,45894.802589,41.267554,45894.802589
3,4,CatBoost is currently the strongest model; eva...,native,none,CatBoost,"{'iterations': 500, 'learning_rate': 0.05, 'de...",completed,47840.730379,18.783108,45894.802589
4,5,Missing values are present; test whether the a...,impute,none,CatBoost,{},completed,45601.015343,35.692789,45601.015343
5,6,CatBoost is currently the strongest model; eva...,impute,none,CatBoost,"{'iterations': 500, 'learning_rate': 0.05, 'de...",completed,47880.001764,17.696332,45601.015343
6,7,Evaluate numerical interactions around the cur...,impute,interactions,CatBoost,{},completed,45482.045323,58.162211,45482.045323
7,8,CatBoost is currently the strongest model; eva...,impute,interactions,CatBoost,"{'iterations': 500, 'learning_rate': 0.05, 'de...",completed,47983.072262,29.174203,45482.045323
8,9,Missing values are present; test whether the a...,native,interactions,CatBoost,{},completed,45889.689661,59.128492,45482.045323
9,10,Explore an untested candidate within the remai...,native,none,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,47491.527633,2.110936,45482.045323


### California Housing - Numeric Regression

The final generalization test uses the numeric-only California Housing dataset.

This dataset contains the same regression target but removes the categorical feature and retains complete numerical observations.

The benchmark tests whether the Senior Agent adapts its experiment planning to a simpler dataset structure.

Because no missing values are present, the agent should not prioritize alternative imputation experiments and can instead allocate more of its budget to model, hyperparameter, and feature engineering decisions.

In [38]:
CALIFORNIA_NUMERIC_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/california_housing_numeric.csv"
CALIFORNIA_NUMERIC_TARGET = "median_house_value"

california_numeric_state = agent(
    CALIFORNIA_NUMERIC_PATH,
    CALIFORNIA_NUMERIC_TARGET,
    max_experiments=MAX_EXPERIMENTS
)

In [39]:
california_numeric_summary = {
    "dataset": "California Housing Numeric",
    "task": california_numeric_state["task"],
    "classification_type": california_numeric_state["classification_type"],
    "metric": california_numeric_state["metric"],
    "validation": california_numeric_state["validation"],
    "n_splits": california_numeric_state["n_splits"],
    "experiments_available": california_numeric_state["experiments_available"],
    "experiment_budget": california_numeric_state["experiment_budget"],
    "experiments_executed": california_numeric_state["experiments_executed"],
    "search_reduction_percent": round(
        (
            1
            - california_numeric_state["experiments_executed"]
            / california_numeric_state["experiments_available"]
        ) * 100,
        2
    ),
    "best_model": california_numeric_state["best_model"],
    "best_preprocessing": california_numeric_state["best_preprocessing"],
    "best_feature_engineering": california_numeric_state["best_feature_engineering"],
    "best_params": california_numeric_state["best_params"],
    "best_score": california_numeric_state["best_score"],
    "best_std": california_numeric_state["best_std"],
    "total_duration_seconds": california_numeric_state["total_duration_seconds"]
}

pd.Series(california_numeric_summary)

dataset                     California Housing Numeric
task                                        regression
classification_type                               None
metric                                            rmse
validation                                       KFold
n_splits                                             5
experiments_available                               24
experiment_budget                                   10
experiments_executed                                10
search_reduction_percent                         58.33
best_model                                    CatBoost
best_preprocessing                              native
best_feature_engineering                  interactions
best_params                                         {}
best_score                                45703.188408
best_std                                   1375.848856
total_duration_seconds                       97.229046
dtype: object

In [40]:
pd.DataFrame(california_numeric_state["decision_history"])

,step,reason,preprocessing,feature_engineering,model,hyperparameters,status,score,duration_seconds,best_score_after_step
0,1,Establish baseline performance for LightGBM.,native,none,LightGBM,{},completed,47850.880361,1.018547,47850.880361
1,2,Establish baseline performance for XGBoost.,native,none,XGBoost,{},completed,47721.563665,1.241496,47721.563665
2,3,Establish baseline performance for CatBoost.,native,none,CatBoost,{},completed,45713.420998,16.815160,45713.420998
3,4,CatBoost is currently the strongest model; eva...,native,none,CatBoost,"{'iterations': 500, 'learning_rate': 0.05, 'de...",completed,47952.772010,8.341228,45713.420998
4,5,Evaluate numerical interactions around the cur...,native,interactions,CatBoost,{},completed,45703.188408,38.982882,45703.188408
5,6,CatBoost is currently the strongest model; eva...,native,interactions,CatBoost,"{'iterations': 500, 'learning_rate': 0.05, 'de...",completed,48029.352721,19.659359,45703.188408
6,7,Explore an untested candidate within the remai...,native,none,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,47634.919186,1.922248,45703.188408
7,8,Explore an untested candidate within the remai...,native,none,XGBoost,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,51423.762079,1.294050,45703.188408
8,9,Explore an untested candidate within the remai...,native,interactions,LightGBM,{},completed,47824.971367,2.871975,45703.188408
9,10,Explore an untested candidate within the remai...,native,interactions,LightGBM,"{'n_estimators': 200, 'learning_rate': 0.05, '...",completed,47618.400981,5.082102,45703.188408


## 20. Benchmark Comparison

The final comparison summarizes how the same Senior Agent behaves across different machine learning problems.

The benchmarks include:

- Adult Income - binary classification with mixed features and missing values
- Wine - multiclass classification with numerical features
- California Housing - regression with mixed features and missing values
- California Housing Numeric - regression with numerical features only

For each dataset, we report the detected task, selected metric, validation strategy, experiment budget, number of executed experiments, search reduction, best observed configuration, validation performance, and total execution time.

This comparison evaluates both predictive performance and the ability of the Senior Agent to adapt its experiment-planning strategy across different dataset characteristics.

In [41]:
benchmark_comparison = pd.DataFrame([
    primary_summary,
    wine_summary,
    california_summary,
    california_numeric_summary
])

benchmark_comparison

,dataset,task,classification_type,metric,validation,n_splits,experiments_available,experiment_budget,experiments_executed,search_reduction_percent,best_model,best_preprocessing,best_feature_engineering,best_params,best_score,best_std,total_duration_seconds
0,Adult Income,classification,binary,roc_auc,StratifiedKFold,5,24,10,10,58.33,CatBoost,native,none,{},0.930631,0.002246,843.473966
1,Wine,classification,multiclass,roc_auc_ovr_macro,StratifiedKFold,5,24,10,10,58.33,CatBoost,native,none,{},0.998847,0.001814,103.446037
2,California Housing,regression,None,rmse,KFold,5,24,10,10,58.33,CatBoost,impute,interactions,{},45482.045323,1289.982782,265.454362
3,California Housing Numeric,regression,None,rmse,KFold,5,24,10,10,58.33,CatBoost,native,interactions,{},45703.188408,1375.848856,97.229046


In [42]:
benchmark_comparison[
    [
        "dataset",
        "task",
        "classification_type",
        "metric",
        "validation",
        "n_splits",
        "experiments_available",
        "experiment_budget",
        "experiments_executed",
        "search_reduction_percent",
        "best_model",
        "best_preprocessing",
        "best_feature_engineering",
        "best_score",
        "best_std",
        "total_duration_seconds"
    ]
]

,dataset,task,classification_type,metric,validation,n_splits,experiments_available,experiment_budget,experiments_executed,search_reduction_percent,best_model,best_preprocessing,best_feature_engineering,best_score,best_std,total_duration_seconds
0,Adult Income,classification,binary,roc_auc,StratifiedKFold,5,24,10,10,58.33,CatBoost,native,none,0.930631,0.002246,843.473966
1,Wine,classification,multiclass,roc_auc_ovr_macro,StratifiedKFold,5,24,10,10,58.33,CatBoost,native,none,0.998847,0.001814,103.446037
2,California Housing,regression,None,rmse,KFold,5,24,10,10,58.33,CatBoost,impute,interactions,45482.045323,1289.982782,265.454362
3,California Housing Numeric,regression,None,rmse,KFold,5,24,10,10,58.33,CatBoost,native,interactions,45703.188408,1375.848856,97.229046


## Notes

The agent now:

- detects binary classification, multiclass classification, and regression tasks;
- inspects the dataset, including missing-value percentages, duplicate rows, and target distribution;
- identifies numerical and categorical features;
- selects the evaluation metric according to the detected task;
- uses `ROC AUC` for binary classification, `ROC AUC OvR Macro` for multiclass classification, and `RMSE` for regression;
- selects the validation strategy according to the detected task;
- uses `StratifiedKFold` for classification and `KFold` for regression;
- adapts the number of cross-validation folds when classification classes are too small;
- keeps preprocessing and feature engineering inside each cross-validation fold;
- builds the complete preprocessing-feature engineering-model-hyperparameter experiment space;
- establishes baseline performance across LightGBM, XGBoost, and CatBoost;
- selects subsequent experiments adaptively according to previous results and dataset characteristics;
- prioritizes promising hyperparameter configurations;
- evaluates alternative preprocessing when missing values are present;
- evaluates feature interactions around promising configurations;
- operates within a fixed experiment budget;
- tracks experiment execution time, status, failures, fold scores, mean score, and standard deviation;
- records the reason behind every experiment selection;
- maintains a complete decision history;
- returns the best observed preprocessing strategy, feature engineering strategy, model, hyperparameters, score, score variability, and decision history.

Version 8 keeps the deliberately small experiment space introduced in Version 7.

With 2 preprocessing strategies, 2 feature engineering strategies, 3 models, and 2 hyperparameter configurations, 24 candidate experiments remain available.

Unlike Version 7, the agent does not evaluate the complete experiment space exhaustively.

With the default experiment budget of 10 and 5-fold cross-validation, the Senior Agent performs at most 50 model fits per dataset instead of the 120 model fits required by the exhaustive Version 7 search.

The experiment-planning policy is deterministic, transparent, and based on dataset inspection and previously observed experiment results.

The complete agent is also evaluated on binary classification, multiclass classification, mixed-type regression, and numeric-only regression datasets.

The architecture remains intentionally simple and rule-based.

Future versions will introduce new components and gradually evolve the architecture.